In [2]:
print("testsing")
import glob

testsing


In [3]:
import json, glob, sys
from pathlib import Path
import pandas as pd
from tqdm import tqdm

sys.path.insert(0, str(Path("..").resolve()))

from ontomap.postprocess.process import eval_preprocess_ir_outputs
from ontomap.evaluation.metrics import evaluation_report

BASE = Path("/vsc-hard-mounts/leuven-data/385/vsc38504/thesis/llms4om/LLMs4OM")
OUTPUTS = BASE / "experiments/outputs/bio-ml/ncit-doid.disease"
DATASET = BASE / "datasets/bio-ml/ncit-doid.disease/om.json"
BASELINE_CSV = BASE / "experiments/results/rag-hybrid-model-results.csv"

with open(DATASET) as f:
    ds = json.load(f)

reference_full = ds["reference"]["equiv"]["full"]
print(f"Reference pairs: {len(reference_full):,}")

Reference pairs: 4,686


In [4]:
def extract_results_from_file_pattern(file_pattern, metrics=["precision", "recall", "f-score"], datasets=["ncit-doid.disease", "omim-ordo.disease", "snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm"]):
    rows = []
    files = sorted(glob.glob(file_pattern), key=lambda f: Path(f).stat().st_mtime)
    print(f'Found {len(files)} files.')
    for f in files:
        with open(f) as fp:
            d = json.load(fp)
        if (d.get('dataset-info', {}).get('ontology-name') not in datasets):
            continue
        row = {
            'model': d.get('model'),
            'task': d.get('dataset-info', {}).get('ontology-name'),
            'encoder': d.get('encoder-id'),
        }
        evaluation_results = d.get('evaluation-results')
        if evaluation_results is not None:
            r = evaluation_results.get('full')
            for metric in metrics: 
                row[metric] = round(r.get(metric), 2)
        else:
            for metric in metrics:
                row[metric] = 'N/A'
        rows.append(row)
    pd.set_option('display.max_colwidth', None)
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=['model', 'task', 'encoder'], keep='last')
    df2 = df.set_index(['model', 'task', 'encoder'])
    df3 = df2.unstack(['task', 'encoder'])
    df3 = df3.reorder_levels(['task', 'encoder', None], axis=1).sort_index(axis=1)
    return df3

## Experiment 1: Embedding Models
**Important note:** N/A means that results are evaluate, just need to be evaluated. NaN means the models still have to be run. 

*notes:*
- Qwen embedding 0.6 B has to be re-run from snomed.body-fma label encoding onwards.
- QWEN embeddung 0.6B and 4B omim-ordo.disease label-children OOM error
- Models still to be run completely: 
    - Embedding gemma 300M
    - NV-embed (mid-tier)
    - gemini embedding 001

In [5]:
retriever_results = extract_results_from_file_pattern('outputs/bio-ml/*/*retrieval*.json', metrics=["recall"])
retriever_results

Found 100 files.


task                            ncit-doid.disease                              \
encoder                                     label label-children label-parent   
                                           recall         recall       recall   
model                                                                           
BERTRetrieval                               90.25           77.4        85.51   
EmbeddingGemma300MRetrieval                  90.1           74.9        83.38   
LlamaNemotronEmbeddingRetrieval              82.8          73.69        87.13   
Qwen3Embedding4BRetrieval                   94.28          79.49        89.74   
Qwen3EmbeddingRetrieval                     90.61          75.63        86.38   
SpecterBERTRetrieval                        90.74          79.58        87.54   
TFIDFRetrieval                              81.48          81.14        79.66   

task                            omim-ordo.disease                              \
encoder                                     label label-children label-parent   
                                           recall         recall       recall   
model                                                                           
BERTRetrieval                               71.49          66.62        70.33   
EmbeddingGemma300MRetrieval                 69.34           68.1        67.11   
LlamaNemotronEmbeddingRetrieval             59.82          60.44        63.59   
Qwen3Embedding4BRetrieval                   73.13          72.83        73.64   
Qwen3EmbeddingRetrieval                     69.74          69.58        69.23   
SpecterBERTRetrieval                        70.89          68.29        68.34   
TFIDFRetrieval                              69.47           69.5        69.04   

task                            snomed-fma.body                              \
encoder                                   label label-children label-parent   
                                         recall         recall       recall   
model                                                                         
BERTRetrieval                             74.67           49.1        63.45   
EmbeddingGemma300MRetrieval               68.52          60.02        60.58   
LlamaNemotronEmbeddingRetrieval           66.62           66.0        80.87   
Qwen3Embedding4BRetrieval                 77.37          61.25        68.11   
Qwen3EmbeddingRetrieval                   69.79          64.37        67.42   
SpecterBERTRetrieval                      52.63           40.7         47.7   
TFIDFRetrieval                              NaN            NaN          NaN   

task                            snomed-ncit.neoplas                 \
encoder                                       label label-children   
                                             recall         recall   
model                                                                
BERTRetrieval                                 79.65          74.71   
EmbeddingGemma300MRetrieval                    74.4          70.16   
LlamaNemotronEmbeddingRetrieval               57.97          56.94   
Qwen3Embedding4BRetrieval                     84.96          82.31   
Qwen3EmbeddingRetrieval                       79.28          75.92   
SpecterBERTRetrieval                          78.55          75.53   
TFIDFRetrieval                                  NaN            NaN   

task                                         snomed-ncit.pharm                 \
encoder                         label-parent             label label-children   
                                      recall            recall         recall   
model                                                                           
BERTRetrieval                          73.05             92.87          90.09   
EmbeddingGemma300MRetrieval            65.64             88.87          86.83   
LlamaNemotronEmbeddingRetrieval        59.91             91.28          88.66   
Qwen3Emb

# Experiment 2: Large language models evaluation

*notes*:
- models still to be run completely: 
    - Gemma 2 2B
    - the instruction variants

In [6]:
pd.set_option('display.max_columns', None)
extract_results_from_file_pattern('outputs/bio-ml/*/*BertRAG*.json', datasets=["ncit-doid.disease", "omim-ordo.disease"])

Found 79 files.


task                  ncit-doid.disease                                  \
encoder                           label                  label-children   
                                f-score precision recall        f-score   
model                                                                     
FalconBertRAG                     81.66     93.92  72.24          81.59   
Gemma2_9BBertRAG                  80.77     94.95  70.27          40.86   
Gemma4_26B_A4BBertRAG             81.44      94.3  71.66          79.04   
LLaMA3BertRAG                     81.68      93.9  72.28          81.65   
LLaMA7BBertRAG                    81.68      93.9  72.28          81.54   
MistralBertRAG                     80.9     94.32  70.83          73.35   
MistralNemoBertRAG                81.68      93.9  72.28          81.69   
Qwen25BertRAG                     81.68      93.9  72.28          81.66   
Qwen25_3BBertRAG                  81.68      93.9  72.28          81.64   
Qwen35_9BBertRAG                    N/A       N/A    N/A            N/A   
VicunaBertRAG                     75.79     91.62  64.62           80.1   

task                                                                  \
encoder                                label-parent                    
                      precision recall      f-score precision recall   
model                                                                  
FalconBertRAG             93.84  72.17        81.61     93.89  72.17   
Gemma2_9BBertRAG          93.23  26.16        77.81     95.64  65.58   
Gemma4_26B_A4BBertRAG     94.13  68.12        68.75     92.53  54.69   
LLaMA3BertRAG              93.9  72.24        81.68      93.9  72.28   
LLaMA7BBertRAG            93.83  72.09        81.68      93.9  72.28   
MistralBertRAG            94.36  59.99        78.68     94.77  67.26   
MistralNemoBertRAG        93.93  72.28        81.68      93.9  72.28   
Qwen25BertRAG             93.92  72.24        81.68      93.9  72.28   
Qwen25_3BBertRAG           93.9  72.22        81.68      93.9  72.28   
Qwen35_9BBertRAG            N/A    N/A          NaN       NaN    NaN   
VicunaBertRAG             93.61   70.0        79.68     93.53   69.4   

task                  omim-ordo.disease                                  \
encoder                           label                  label-children   
                                f-score precision recall        f-score   
model                                                                     
FalconBertRAG                      58.3     88.79   43.4          58.27   
Gemma2_9BBertRAG                   54.0     89.17  38.73          14.04   
Gemma4_26B_A4BBertRAG             58.29     88.83  43.38           58.1   
LLaMA3BertRAG                      58.3     88.79   43.4          58.24   
LLaMA7BBertRAG                     58.3     88.79   43.4          58.15   
MistralBertRAG                    57.84     88.91  42.86          48.28   
MistralNemoBertRAG                  NaN       NaN    NaN            NaN   
Qwen25BertRAG                       NaN       NaN    NaN            NaN   
Qwen25_3BBertRAG                    NaN       NaN    NaN            NaN   
Qwen35_9BBertRAG                   58.3     88.79   43.4          58.23   
VicunaBertRAG                     54.54     87.14  39.69          56.26   

task                                                                  
encoder                                label-parent                   
                      precision recall      f-score precision recall  
model                                                                 
FalconBertRAG             88.73  43.38         58.3     88.79   43.4  
Gemma2_9BBertRAG          84.07   7.66        51.69     89.88  36.28  
Gemma4_26B_A4BBertRAG      89.2  43.08        57.41     89.09  42.35  
LLaMA3BertRAG             88.72  43.35         58.3     88.79   43.4  
LLaMA7BBertRAG            88.66  43.27         58.3     88.79   43.4  
MistralBertRAG            83